# Notebook 0: Security Corpus Collection

**Runtime: CPU (no GPU needed)**

This notebook collects raw security text from public sources and writes
it to Google Drive as JSONL. No GPU required — save your compute hours
for Notebooks 2 and 3.

## Sources
1. **MITRE ATT&CK** — Techniques, procedures, threat groups (STIX JSON)
2. **SigmaHQ** — Detection rules with severity labels and domain tags
3. **NVD CVEs** — Vulnerability descriptions via NVD API
4. **CISA Advisories** — Known exploited vulnerabilities catalog

## Output
```
FedDAPT/corpus/raw/security_corpus_raw.jsonl
FedDAPT/corpus/raw/collection_report.json
```

---
## 0. Setup

In [1]:
!pip -q install requests pyyaml

In [2]:
import os, json, glob, time, random, hashlib
from collections import Counter
from datetime import datetime
import requests
import yaml

print(f'Started: {datetime.now().isoformat()}')

Started: 2026-07-16T19:57:02.959467


In [3]:

import os, tempfile
def _load_dotenv(path='.env'):
    # Minimal .env loader (no dependency); real environment vars take precedence.
    try:
        with open(path) as _f:
            for _line in _f:
                _line = _line.strip()
                if not _line or _line.startswith('#') or '=' not in _line:
                    continue
                _k, _v = _line.split('=', 1)
                os.environ.setdefault(_k.strip(), _v.strip().strip('\'"'))
    except FileNotFoundError:
        pass
def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False
_load_dotenv()   # pull NVD_API_KEY / FEDDAPT_ROOT from .env if present
if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    PROJECT_ROOT = os.environ.get('FEDDAPT_ROOT', '/content/drive/MyDrive/FedDAPT')
else:
    # PyCharm / remote GPU: set FEDDAPT_ROOT to your data dir, e.g.  export FEDDAPT_ROOT=/data/fedapt
    PROJECT_ROOT = os.environ.get('FEDDAPT_ROOT', os.path.abspath('./FedDAPT'))
SCRATCH = os.environ.get('FEDDAPT_WORK', os.path.join(tempfile.gettempdir(), 'fedapt_work'))
os.makedirs(PROJECT_ROOT, exist_ok=True); os.makedirs(SCRATCH, exist_ok=True)
print('PROJECT_ROOT =', PROJECT_ROOT, '| SCRATCH =', SCRATCH)
RAW_DIR = f'{PROJECT_ROOT}/corpus/raw'
WORK_DIR = SCRATCH

for d in [RAW_DIR, WORK_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Output: {RAW_DIR}')

ValueError: mount failed

In [ ]:
# Collector that accumulates all documents with stats
all_docs = []
collection_stats = {}

def doc_id(text):
    """Generate a deterministic short ID from text content."""
    return hashlib.md5(text.encode('utf-8')).hexdigest()[:12]

---
## 1. MITRE ATT&CK (STIX)

Downloads the full Enterprise ATT&CK knowledge base as STIX 2.1 JSON.
Extracts techniques, procedure examples, threat group profiles,
software descriptions, and mitigations.

In [ ]:
ATTACK_URL = 'https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json'
ATTACK_CACHE = f'{RAW_DIR}/enterprise-attack.json'

if os.path.exists(ATTACK_CACHE):
    print('Loading cached ATT&CK STIX...')
    with open(ATTACK_CACHE, 'r') as f:
        attack_data = json.load(f)
else:
    print('Downloading ATT&CK STIX from GitHub...')
    r = requests.get(ATTACK_URL, timeout=120)
    r.raise_for_status()
    attack_data = r.json()
    with open(ATTACK_CACHE, 'w') as f:
        json.dump(attack_data, f)
    print(f'Cached to {ATTACK_CACHE}')

print(f'STIX objects: {len(attack_data.get("objects", []))}')

In [ ]:
attack_docs = []
attack_counts = Counter()

for obj in attack_data.get('objects', []):
    if obj.get('revoked') or obj.get('x_mitre_deprecated'):
        continue

    desc = obj.get('description', '')
    if not desc or len(desc.strip()) < 20:
        continue

    obj_type = obj.get('type', '')
    name = obj.get('name', '')
    ext_refs = obj.get('external_references', [{}])
    ext_id = ext_refs[0].get('external_id', '') if ext_refs else ''

    if obj_type == 'attack-pattern':
        attack_docs.append({
            'text': f'MITRE ATT&CK Technique {ext_id} ({name}): {desc}',
            'id': f'attack-tech-{ext_id}',
            'source': 'mitre_attack',
            'subdomain': 'cti',
            'category': 'technique',
        })
        attack_counts['techniques'] += 1

    elif obj_type == 'relationship' and obj.get('relationship_type') == 'uses':
        attack_docs.append({
            'text': f'MITRE ATT&CK Procedure Example: {desc}',
            'id': f'attack-proc-{doc_id(desc)}',
            'source': 'mitre_attack',
            'subdomain': 'cti',
            'category': 'procedure',
        })
        attack_counts['procedures'] += 1

    elif obj_type == 'intrusion-set':
        attack_docs.append({
            'text': f'MITRE ATT&CK Threat Group {name}: {desc}',
            'id': f'attack-group-{name.replace(" ", "_").replace("/", "_")}',
            'source': 'mitre_attack',
            'subdomain': 'cti',
            'category': 'group',
        })
        attack_counts['groups'] += 1

    elif obj_type == 'malware' and desc:
        attack_docs.append({
            'text': f'MITRE ATT&CK Malware {name}: {desc}',
            'id': f'attack-malware-{name.replace(" ", "_").replace("/", "_")}',
            'source': 'mitre_attack',
            'subdomain': 'cti',
            'category': 'malware',
        })
        attack_counts['malware'] += 1

    elif obj_type == 'tool' and desc:
        attack_docs.append({
            'text': f'MITRE ATT&CK Tool {name}: {desc}',
            'id': f'attack-tool-{name.replace(" ", "_").replace("/", "_")}',
            'source': 'mitre_attack',
            'subdomain': 'cti',
            'category': 'tool',
        })
        attack_counts['tools'] += 1

    elif obj_type == 'course-of-action' and desc:
        attack_docs.append({
            'text': f'MITRE ATT&CK Mitigation {name}: {desc}',
            'id': f'attack-mitigation-{ext_id or doc_id(desc)}',
            'source': 'mitre_attack',
            'subdomain': 'cti',
            'category': 'mitigation',
        })
        attack_counts['mitigations'] += 1

all_docs.extend(attack_docs)
collection_stats['mitre_attack'] = dict(attack_counts)
collection_stats['mitre_attack']['total'] = len(attack_docs)

print(f'\nMITRE ATT&CK: {len(attack_docs)} documents')
for k, v in attack_counts.most_common():
    print(f'  {k}: {v}')

---
## 2. SigmaHQ Detection Rules

Clones the SigmaHQ repository and parses all YAML detection rules.
Each rule gets a subdomain label (endpoint, network, cloud, general)
based on its logsource field.

In [ ]:
SIGMA_PATH = f'{WORK_DIR}/sigma'

if not os.path.exists(SIGMA_PATH):
    print('Cloning SigmaHQ (this takes 1-2 minutes)...')
    !git clone --depth 1 https://github.com/SigmaHQ/sigma.git {SIGMA_PATH}
else:
    print(f'Sigma already at {SIGMA_PATH}')

yaml_files = glob.glob(f'{SIGMA_PATH}/**/*.yml', recursive=True)
print(f'Found {len(yaml_files)} YAML files')

In [ ]:
sigma_docs = []
sigma_domains = Counter()
sigma_levels = Counter()
sigma_errors = 0

for fpath in yaml_files:
    try:
        with open(fpath, 'r', encoding='utf-8') as f:
            rule = yaml.safe_load(f)
        if not isinstance(rule, dict):
            continue

        title = rule.get('title', '')
        desc = rule.get('description', '')
        level = rule.get('level', 'unknown')
        status = rule.get('status', 'unknown')
        tags = rule.get('tags', [])
        logsource = rule.get('logsource', {})
        logsource_str = str(logsource).lower()

        if not (title or desc):
            continue

        # Classify subdomain from logsource
        if any(k in logsource_str for k in ['process', 'sysmon', 'powershell', 'windows', 'endpoint', 'registry', 'image_load']):
            subdomain = 'endpoint'
        elif any(k in logsource_str for k in ['firewall', 'network', 'dns', 'proxy', 'zeek', 'suricata', 'netflow']):
            subdomain = 'network'
        elif any(k in logsource_str for k in ['aws', 'azure', 'gcp', 'cloud', 'o365', 'okta', 'google_workspace']):
            subdomain = 'cloud'
        else:
            subdomain = 'general'

        # Build text
        text = f'Sigma Detection Rule [{level.upper()}]: {title}.'
        if desc:
            text += f' {desc}'

        # Add ATT&CK tags if present
        attack_tags = [t for t in tags if isinstance(t, str) and t.startswith('attack.')]
        if attack_tags:
            text += f' MITRE ATT&CK: {', '.join(attack_tags)}.'

        # Add logsource context
        category = logsource.get('category', '')
        product = logsource.get('product', '')
        service = logsource.get('service', '')
        logsource_parts = [p for p in [category, product, service] if p]
        if logsource_parts:
            text += f' Logsource: {', '.join(logsource_parts)}.'

        rule_id = os.path.basename(fpath).replace('.yml', '')
        sigma_docs.append({
            'text': text,
            'id': f'sigma-{rule_id}',
            'source': 'sigma',
            'subdomain': subdomain,
            'level': level,
        })

        sigma_domains[subdomain] += 1
        sigma_levels[level] += 1

    except Exception:
        sigma_errors += 1
        continue

all_docs.extend(sigma_docs)
collection_stats['sigma'] = {
    'total': len(sigma_docs),
    'parse_errors': sigma_errors,
    'by_domain': dict(sigma_domains),
    'by_level': dict(sigma_levels),
}

print(f'\nSigma: {len(sigma_docs)} rules ({sigma_errors} parse errors)')
print(f'By domain:')
for d, c in sigma_domains.most_common():
    print(f'  {d}: {c}')
print(f'By severity:')
for l, c in sigma_levels.most_common():
    print(f'  {l}: {c}')

---
## 3. NVD CVEs

Fetches vulnerability descriptions from the National Vulnerability Database API.
Filters out reserved/rejected placeholders.

**Note:** The NVD API is rate-limited. Without an API key you get ~5 requests/30s.
With a free API key (register at https://nvd.nist.gov/developers/request-an-api-key)
you get ~50 requests/30s.

In [ ]:

def _get_secret(name, default=''):
    try:
        from google.colab import userdata
        return (userdata.get(name) or default).strip()
    except Exception:
        return os.environ.get(name, default).strip()
NVD_API_KEY = _get_secret('NVD_API_KEY')  # Colab: Secrets panel; else env var NVD_API_KEY
NVD_MAX_PAGES = 10  # ~2000 CVEs per page, 10 pages = ~20k CVEs

def fetch_nvd_cves(api_key=None, max_pages=5, results_per_page=2000):
    """Fetch CVE descriptions from NVD API v2.0."""
    base_url = 'https://services.nvd.nist.gov/rest/json/cves/2.0'
    headers = {}
    if api_key:
        headers['apiKey'] = api_key

    docs = []
    start_index = 0
    reject_patterns = ['reserved', 'rejected', 'not yet provided', '** disputed']

    for page in range(max_pages):
        print(f'  Page {page+1}/{max_pages} (startIndex={start_index})...')
        params = {'startIndex': start_index, 'resultsPerPage': results_per_page}

        try:
            r = requests.get(base_url, headers=headers, params=params, timeout=120)
            r.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f'  Request failed: {e}')
            break

        data = r.json()
        vulns = data.get('vulnerabilities', [])

        for v in vulns:
            cve = v.get('cve', {})
            cve_id = cve.get('id', '')

            # Find English description
            desc_en = ''
            for d in cve.get('descriptions', []):
                if d.get('lang') == 'en':
                    desc_en = d.get('value', '')
                    break

            if not desc_en:
                continue

            # Skip placeholders
            desc_lower = desc_en.lower()
            if any(p in desc_lower for p in reject_patterns) and len(desc_en) < 150:
                continue

            # Extract CVSS if available
            metrics = cve.get('metrics', {})
            cvss_str = ''
            for version in ['cvssMetricV31', 'cvssMetricV30', 'cvssMetricV2']:
                if version in metrics and metrics[version]:
                    score = metrics[version][0].get('cvssData', {}).get('baseScore', '')
                    severity = metrics[version][0].get('cvssData', {}).get('baseSeverity', '')
                    if score:
                        cvss_str = f' CVSS: {score} ({severity}).'
                    break

            # Extract CWE if available
            cwe_str = ''
            weaknesses = cve.get('weaknesses', [])
            for w in weaknesses:
                for wd in w.get('description', []):
                    if wd.get('lang') == 'en' and wd.get('value', '').startswith('CWE-'):
                        cwe_str = f' Weakness: {wd["value"]}.'
                        break
                if cwe_str:
                    break

            text = f'CVE {cve_id}: {desc_en}{cvss_str}{cwe_str}'

            docs.append({
                'text': text,
                'id': f'nvd-{cve_id}',
                'source': 'nvd',
                'subdomain': 'vulnerability',
            })

        total = data.get('totalResults', 0)
        start_index += results_per_page
        if start_index >= total:
            print(f'  Reached end of results ({total} total)')
            break

        # Rate limit
        wait = 0.8 if api_key else 6.0
        time.sleep(wait)

    return docs


print('Fetching NVD CVEs...')
cve_docs = fetch_nvd_cves(api_key=NVD_API_KEY, max_pages=NVD_MAX_PAGES)

all_docs.extend(cve_docs)
collection_stats['nvd'] = {'total': len(cve_docs)}

print(f'\nNVD: {len(cve_docs)} CVE documents')
if cve_docs:
    print(f'Sample: {cve_docs[0]["text"][:150]}...')

---
## 4. CISA Known Exploited Vulnerabilities

CISA publishes a catalog of known exploited vulnerabilities (KEV)
as a JSON feed. These are high-value entries because they represent
vulnerabilities actively used in the wild.

In [ ]:
KEV_URL = 'https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json'

print('Fetching CISA KEV catalog...')
try:
    r = requests.get(KEV_URL, timeout=60)
    r.raise_for_status()
    kev_data = r.json()

    kev_docs = []
    for vuln in kev_data.get('vulnerabilities', []):
        cve_id = vuln.get('cveID', '')
        vendor = vuln.get('vendorProject', '')
        product = vuln.get('product', '')
        vuln_name = vuln.get('vulnerabilityName', '')
        desc = vuln.get('shortDescription', '')
        action = vuln.get('requiredAction', '')
        date_added = vuln.get('dateAdded', '')

        if not desc:
            continue

        text = f'CISA Known Exploited Vulnerability {cve_id}: {vuln_name}.'
        text += f' Vendor: {vendor}, Product: {product}.'
        text += f' {desc}'
        if action:
            text += f' Required action: {action}.'
        if date_added:
            text += f' Added to KEV catalog: {date_added}.'

        kev_docs.append({
            'text': text,
            'id': f'cisa-kev-{cve_id}',
            'source': 'cisa_kev',
            'subdomain': 'vulnerability',
        })

    all_docs.extend(kev_docs)
    collection_stats['cisa_kev'] = {'total': len(kev_docs)}
    print(f'CISA KEV: {len(kev_docs)} entries')

except Exception as e:
    print(f'CISA KEV fetch failed: {e}')
    kev_docs = []
    collection_stats['cisa_kev'] = {'total': 0, 'error': str(e)}

---
## 5. MITRE CAR (Cyber Analytics Repository)

CAR contains analytics mapped to ATT&CK with natural language
descriptions of detection logic. Good supplemental data for
alert triage and detection engineering context.

In [ ]:
CAR_URL = 'https://raw.githubusercontent.com/mitre-attack/car/master/analytics/'
CAR_INDEX = 'https://api.github.com/repos/mitre-attack/car/contents/analytics'

print('Fetching MITRE CAR analytics...')
try:
    r = requests.get(CAR_INDEX, timeout=30)
    r.raise_for_status()
    car_files = [item for item in r.json() if item['name'].endswith('.yaml')]

    car_docs = []
    for i, item in enumerate(car_files):
        try:
            resp = requests.get(item['download_url'], timeout=15)
            resp.raise_for_status()
            analytic = yaml.safe_load(resp.text)

            if not isinstance(analytic, dict):
                continue

            title = analytic.get('title', '')
            desc = analytic.get('description', '')
            car_id = analytic.get('id', '')

            if not desc or len(desc) < 20:
                continue

            # Extract ATT&CK coverage
            coverage = analytic.get('coverage', [])
            techniques = []
            for c in coverage:
                tech = c.get('technique', '')
                if tech:
                    techniques.append(tech)

            text = f'MITRE CAR Analytic {car_id}: {title}. {desc}'
            if techniques:
                text += f' Covers ATT&CK techniques: {', '.join(techniques)}.'

            car_docs.append({
                'text': text,
                'id': f'car-{car_id}',
                'source': 'mitre_car',
                'subdomain': 'detection',
            })

        except Exception:
            continue

        if (i + 1) % 20 == 0:
            print(f'  Processed {i+1}/{len(car_files)} CAR files')
            time.sleep(0.5)  # GitHub rate limit

    all_docs.extend(car_docs)
    collection_stats['mitre_car'] = {'total': len(car_docs)}
    print(f'MITRE CAR: {len(car_docs)} analytics')

except Exception as e:
    print(f'MITRE CAR fetch failed: {e}')
    collection_stats['mitre_car'] = {'total': 0, 'error': str(e)}

---
## 6. Corpus Summary + Export

In [ ]:
import numpy as np

print(f'\n{"="*60}')
print(f'RAW CORPUS SUMMARY')
print(f'{"="*60}')
print(f'Total documents: {len(all_docs)}')

source_counts = Counter(d['source'] for d in all_docs)
print(f'\nBy source:')
for s, c in source_counts.most_common():
    print(f'  {s}: {c}')

subdomain_counts = Counter(d.get('subdomain', 'unknown') for d in all_docs)
print(f'\nBy subdomain:')
for s, c in subdomain_counts.most_common():
    print(f'  {s}: {c}')

# Text length stats
lengths = [len(d['text']) for d in all_docs]
word_counts = [len(d['text'].split()) for d in all_docs]
print(f'\nText length:')
print(f'  Mean: {np.mean(lengths):.0f} chars, {np.mean(word_counts):.0f} words')
print(f'  Median: {np.median(lengths):.0f} chars')
print(f'  Min: {min(lengths)} chars, Max: {max(lengths)} chars')

In [ ]:
# Need numpy for stats above
import numpy as np

# ============================================================
# Write raw corpus JSONL to Drive
# ============================================================
output_path = f'{RAW_DIR}/security_corpus_raw.jsonl'

with open(output_path, 'w', encoding='utf-8') as f:
    for doc in all_docs:
        f.write(json.dumps(doc, ensure_ascii=False) + '\n')

file_size_mb = os.path.getsize(output_path) / 1e6
print(f'\nWritten: {output_path}')
print(f'File size: {file_size_mb:.1f} MB')
print(f'Documents: {len(all_docs)}')

In [ ]:
# ============================================================
# Save collection report for paper methodology
# ============================================================
collection_report = {
    'timestamp': datetime.now().isoformat(),
    'total_documents': len(all_docs),
    'output_file': output_path,
    'file_size_mb': round(file_size_mb, 1),
    'sources': collection_stats,
    'source_distribution': dict(source_counts),
    'subdomain_distribution': dict(subdomain_counts),
    'text_stats': {
        'mean_chars': round(float(np.mean(lengths)), 0),
        'median_chars': round(float(np.median(lengths)), 0),
        'min_chars': int(min(lengths)),
        'max_chars': int(max(lengths)),
        'mean_words': round(float(np.mean(word_counts)), 0),
    },
    # NOTE: removal_rate_by_source is populated in Notebook 1 after curation
    'data_sources_cited': [
        'MITRE ATT&CK Enterprise v15+ (STIX 2.1)',
        'SigmaHQ Detection Rules (GitHub)',
        'NIST National Vulnerability Database API v2.0',
        'CISA Known Exploited Vulnerabilities Catalog',
        'MITRE Cyber Analytics Repository (CAR)',
    ],
}

report_path = f'{RAW_DIR}/collection_report.json'
with open(report_path, 'w') as f:
    json.dump(collection_report, f, indent=2)

print(f'\nCollection report: {report_path}')
print(json.dumps(collection_report, indent=2))

---
## Next Steps

Raw corpus is on Drive at:
```
FedDAPT/corpus/raw/security_corpus_raw.jsonl
FedDAPT/corpus/raw/collection_report.json
```

Open **Notebook 1** on an **A100 runtime** to curate this data:
- The notebook will load `security_corpus_raw.jsonl` from Drive
- Run NeMo Curator pipeline (cleaning, filtering, dedup, PII redaction)
- Output curated client splits for Notebook 2 (training pipeline: federated DAPT → analysis)

### To add more data later
You can re-run this notebook with additional sources and append to
the JSONL. Then re-run Notebook 1 to re-curate. Notebook 2 (training pipeline: federated DAPT → analysis) auto-loads
whatever curated data is on Drive.

### Sources to consider adding
- [ ] The DFIR Report (manual download, HTML scrape of incident timelines)
- [ ] CISA alerts and advisories (RSS/HTML scrape)
- [ ] CIS Benchmarks (PDF extraction — would need Notebook 1 for this)
- [ ] Security vendor blogs (CrowdStrike, Mandiant, Talos)
- [ ] Exploit-DB descriptions
- [ ] HuggingFace security datasets (search for cybersecurity tag)